## Blackjack classes

This program has classes that will create a game of Blackjack, a player for the game, and train a Qtable to play optimally. Insurance is disabled by default in training and evaluating.

In [2]:
# Import modules
import numpy as np
import random
import os
import sys
from contextlib import redirect_stdout
from collections import defaultdict
import dill as pk
import matplotlib.pyplot as plt

In [3]:
class Blackjack():
    """
    This class holds the environment that plays blackjack. 
     Attributes:
         players (list): Players that are playing
         min_bet (float or int): Minimum bet players can make
         do_[special action] (bool): Select whether to enable special actions (split, double down, insurance, and surrender)
         wagers (bool): Whether to have cash wagers
         verbose (bool): Whether to print game narration
    """
    def __init__(self, players, min_bet=10, do_split_pairs=True, do_double_down=True, do_insurance=True, do_surrender=True, wagers=True, verbose=False):
        self.round = 0
        self.turn = 0
        self.bet_min = min_bet

        if isinstance(players, list):
            self.players = players
        else:
            print('PLEASE ENTER PLAYERS AS A LIST OF PLAYER OBJECTS WITH DEALER AS FIRST PLAYER')
        if len(players) > 1:
            self.player_count = len(players)
        else:
            print('DEALER CANNOT PLAY AGAINST THEMSELVES, ADD SOME PLAYERS')
        self.dealer = players[0]
        
        self.winners = [0] * self.player_count
        self.dealer_done = False
        
        self.do_split_pairs = do_split_pairs
        self.do_double_down = do_double_down
        self.do_insurance = do_insurance
        self.do_surrender = do_surrender
        
        self.split_pairs = False
        self.wagers = wagers
        self.deck = Deck()
        self.verbose = verbose
   
        for player in self.players:
            player.deck = self.deck
            player.game = self  # give every player a reference to the game
        self.max_bet = min_bet * 100

    def play_round(self):
        """
        Play one round of blackjack. Includes player resetting, wagers, split logic handling, and resolution/payout of game
        """
        for player in self.players:
            player.reset()
        def display_card_value(card):
            if isinstance(card, int):
                return card
            elif card in ['J', 'Q', 'K']:
                return 10
            elif card == 'Ace':
                return 11
        round_number = self.round
        if self.wagers:
            for player in self.players:
                player.turn(round_number=0, dealer_card=None)
            else:
                pass
        else:
            for player in self.players:
                player.turn(round_number=0, dealer_card=None)
        if self.verbose:
            print(f"The dealer's hand is: {self.dealer.hand[0]} with a value of {display_card_value(self.dealer.hand[0])} and has one hidden card\n")
    
        i = 1
        while i < len(self.players):
            player = self.players[i]
            if not player.done:
                player.turn(round_number=1, dealer_card=self.dealer.hand[0])
                if hasattr(player, 'split_obj') and player.split_obj is not None:
                    split_player = player.split_obj
                    split_player.game = self
                    self.players.insert(i + 1, split_player)
                    player.split_obj = None
            i += 1
        if self.verbose:
            print(f'Dealer reveals full hand: {self.dealer.hand} with {self.dealer.points} points\n')
        self.deck.count_cards(self.dealer.hand[1])
        while not self.dealer.done:
            self.dealer.turn(round_number=1, dealer_card=self.dealer.hand[0])
        self.resolve()
        self.payout()
        return (self.winners[1:], self.rewards[1:])

    def resolve(self):
        """
        Resolve a game of blackjack. Check every case and assign win values and reward values for training, printing, and wagering.

        Return (list): winners and rewards list for each player, but not dealer
        """
        self.rewards = [0] * len(self.players)
        self.winners = [0] * len(self.players)
        dealer_points = self.dealer.points
        dealer_blackjack = (dealer_points == 21 and len(self.dealer.hand) == 2)

        for i in range(1, len(self.players)):
            player = self.players[i]

            player_blackjack = (player.points == 21 and len(player.hand) == 2 and not player.splitted and '_split' not in player.name)
            if player.busted:
                self.winners[i] = self.rewards[i] = -1
            elif player.surrendered:
                self.winners[i] = self.rewards[i] =-0.5
            elif player_blackjack and dealer_blackjack:
                self.winners[i] =self.rewards[i] = 0
            elif player_blackjack:
                self.winners[i] = self.rewards[i] =1.5
            elif dealer_blackjack and player.win_insurance:
                self.winners[i] = -1
                self.rewards[i] = 0
            elif dealer_blackjack:
                self.winners[i] = self.rewards[i] =-1
            elif self.dealer.busted:
                self.winners[i] = self.rewards[i] =1
            elif player.points > dealer_points:
                self.winners[i] = self.rewards[i] =1
            elif player.points == dealer_points:
                self.winners[i] = self.rewards[i] =0
            else:
                self.winners[i] = self.rewards[i] = -1
            if self.winners[i] == -1 and player.insured and not player.win_insurance:
                self.rewards[i] = -1.5
            if self.verbose:
                if self.winners[i] == -1:
                    print(f'{player.name} has lost with {player.points} points vs {dealer_points} points')
                elif self.winners[i] == 0:
                    print(f'{player.name} has tied with {player.points} points vs {dealer_points} points')
                elif self.winners[i] == 1 or self.winners[i] == 1.5:
                    print(f'{player.name} has won with {player.points} points vs {dealer_points} points')
                elif self.winners[i] == 0.5:
                    print(f'{player.name} has surrendered and lost half of their wager)')
        self.aggregate_split_rewards()
        return (self.winners[1:], self.rewards[1:])
            
    def payout(self):
        """
        Payout each player based on win value. Handles split and insurance logic.
        """
        to_pop = []
        for i in range(1, (len(self.players))):  
            player = self.players[i]
            result = self.winners[i]
            payout = player.calc_winnings(result)

            if getattr(player, 'parent', None) is not None:
                to_pop.append(player)
                player.root_player.bankroll += payout
                if player.insured:
                    self.insurance_reward(player)
            else:
                player.bankroll += payout
                player.adjust_wager()
                if player.insured:
                    self.insurance_reward(player)
                    
        for pl in to_pop:
            self.players.remove(pl)

    def check_surrender(self, player):
        """
        Check if player is currently able to surrender.

        Args:
            Player (object): Which player to check for

        Returns (bool): True if able to surrender, False if not
        """
        if len(player.hand) == 2 and not player.done and self.do_surrender:
            return True
        else:
            return False

    def check_insurance(self, player):
        """
        Check if player is currently able to insure.

        Args:
            Player (object): Which player to check for

        Returns (bool): True if able to insure, False if not
        """
        if self.dealer.hand[0] == 'Ace' and len(player.hand) == 2 and not player.done and self.do_insurance and not player.insured:
            return True
        else:
            return False

    def check_split(self, player):
        """
        Check if player is currently able to split.

        Args:
            Player (object): Which player to check for

        Returns (bool): True if able to split, False if not
        """
        if not (len(player.hand) == 2 and not player.done and self.do_split_pairs and player.root_player.bankroll - player.wager >= 0):
            return False
        def card_value(c):
        # Helper function
            if c in ['J', 'Q', 'K']:
                return 10
            elif c == 'Ace':
                return 11
            else:
                return int(c)
        return card_value(player.hand[0]) == card_value(player.hand[1])

    def check_double_down(self, player):
        """
        Check if player is currently able to double down.

        Args:
            Player (object): Which player to check for

        Returns (bool): True if able to double down, False if not
        """
        if len(player.hand) == 2 and not player.done and player.root_player.bankroll - player.wager >= 0 and self.do_double_down:
            return True
        else:
            return False

    def get_legal_actions(self, player):
        """
        Return a list of legal actions, based on boolean values of check functions
        """
        legal_actions = ['hit', 'stand']
        if self.check_surrender(player):
            legal_actions.append('surrender')
        if self.check_insurance(player):
            legal_actions.append('insurance')
        if self.check_double_down(player):
            legal_actions.append('double_down')
        if self.check_split(player):
            legal_actions.append('split_pairs')        
        return legal_actions

    def insurance_reward(self, player):
        """
        Handle insurance reward and payout logic.
        """
        if self.dealer.points == 21 and len(self.dealer.hand) == 2:
            if self.verbose:
                print(f'{player.name} has won their insurance bet and gains ${player.insurance_bet}')
            player.root_player.bankroll += 3*player.insurance_bet
            player.win_insurance = True
            player.done = True
        else:
            if self.verbose:
                print(f'{player.name} has lost their insurance bet and loses ${player.insurance_bet}')

    def aggregate_split_rewards(self):
        """
        Handle split reward logic (solely for training)
        """
        for i in range(1, len(self.players)):
            player = self.players[i]
            if '_split' in player.name:
                original_name = player.name.replace('_split', '')
                for j in range(1, len(self.players)):
                    if self.players[j].name == original_name:
                        self.rewards[j] += self.rewards[i]
                        self.rewards[i] = 0
                        break


In [5]:
class Player():
    """
    Player/Model class that will play the game of Blackjack
    
    Attributes:
        is_dealer (bool): Checks whether the instance is the dealer. Only one dealer per game.
        default_perc (float): Unused
        min_bet (float): Minimum allowed bet. Must be same value passed into the Blackjack class instance.
        initial_buyin (float): Initial amount in the players bankroll
        name (str): Name of the player
        dec_func (function): Function to be used to make decisions (ideally taken from trained model)
        count_cards (bool): Enable or disable card counting for the player instance
        game (class): Class for the blackjack game
        verbose (bool): Whether to print game narration
        

    """
    def __init__(self, is_dealer=False, default_perc = 0.001, min_bet=10, initial_buyin=50000, name='Bucky', dec_func=None, count_cards=True, game=Blackjack, verbose=True):
        self.name = name
        self.points = 0
        self.hand = []
        self.busted = False
        self.done = False
        self.dec_func = dec_func
        self.is_dealer = is_dealer

        self.bankroll = initial_buyin
        self.wager = 0
        if not is_dealer:
            self.default_wager = min_bet
        else:
            self.default_wager = 0
        self.deck = None
        self.game = None  # set by Blackjack.__init__ and when split hands are created

        self.count_cards_val = count_cards
        self.min_bet = min_bet
        self.verbose = verbose

        self.surrendered = False
        self.double_downed = False
        self.splitted = False
        self.insured = False

        self.win_insurance = False
        self.history = []
        self.max_bet = min_bet * 100
        
    def __str__(self):
        if self.verbose:
            return f'Player {self.name} has {self.points} points with a hand of {self.hand}\n'
            
    @property
    def root_player(self):
        """
        Always points to the original funding player's bankroll.
        """
        root = self
        while getattr(root, 'parent', None) is not None:
            root = root.parent
        return root

    def get_card_value(self, card):
        """
        Get the value a card
        Args:
            card (str or int): card to be checked
        Returns int of card value
        """
        if isinstance(card, int):
            return card
        elif card in ['J', 'Q', 'K']:
            return 10
        elif card == 'Ace':
            self.ace_in_hand = True
            return self.hard_soft_ace()
        else:
            print(card)
            raise ValueError('Check code')

    def calc_points(self):
        """
        Dynamically calculate the point total
        """
        p = 0
        aces = 0
        for c in self.hand:
            if c == 'Ace':
                aces += 1
            else:
                p += self.get_card_value(c)
    
        # Treat all aces as 1 initially
        p += aces
    
        # If we have an ace and can add 10 without busting, it's a soft hand
        if aces > 0 and p + 10 <= 21:
            p += 10

        return p

    def draw(self, display=True):
        """
        Draw a random card
        Args:
            display (bool): Whether to display the selected card (false for second dealer card)
        """
        draw = self.deck.draw(count=self.count_cards_val)
        self.hand.append(draw)
        self.points = self.calc_points()
        if display and self.verbose:
            print(f'A {draw} has been drawn for player {self.name}\nTotal: {self.points}')
        
    def turn(self, round_number, dealer_card, dealer=False):
        """
        Do a single turn of blackjack. Use the decision function if one is provided, if not program defaults to user input. Includes
        a safegaurd that will stop illegal actions. If an illegal action is stopped for a trained decision function, next highest
        value is defaulted to recursively. If not value is found default to hit or stand. Check if busted after each turn.
        
        Args:
            round_number (int): Will be 0 or 1, the first turn has different logic
            dealer_card (int or str): Dealer's up card, used in decision function
            dealer (bool): If the turn is being taken by the dealer or not
        """
        if self.is_dealer:
            dealer = True
        if not self.busted:
            if round_number == 0:
                if dealer:
                    self.draw()
                    self.draw(display=False)
                else:
                    self.make_wager()
                    self.draw()
                    self.draw()
            else:
                if dealer:
                    if self.points <= 16:
                        self.draw()
                    elif self.points == 17:
                        # Calculate the absolute minimum hard points
                        hard_total = sum([self.get_card_value(c) for c in self.hand if c != 'Ace']) + self.hand.count('Ace')
                        # If the hard total is exactly 7 and there is an Ace, it's a soft 17
                        is_soft_17 = (hard_total == 7 and 'Ace' in self.hand)
                
                        if is_soft_17:
                            self.draw()
                        else:
                            self.done = True
                    else:
                        self.done = True
                        
                else:
                    while True:
                            legal_actions = self.game.get_legal_actions(self)
                        else:
                            legal_actions = ['hit', 'stand']

                        if self.dec_func:
                            dec = self.dec_func(self, self.points, self.hand, dealer_card, self.deck.true_count)
                        else:
                            dec = input(f'Hit or Stand ({self.name})')

                        dec = dec.lower().strip()
                        if dec not in legal_actions:
                            if self.verbose:
                                print(f'[SAFEGUARD] "{dec}" is not legal for {self.name} '
                                      f'(hand={self.hand}, legal={legal_actions}). Defaulting to hit/stand.')
                            ranking = getattr(self.dec_func, '_last_action_ranking', []) if self.dec_func else []
                            # pick the highest-ranked action that is actually legal
                            fallback = next((a for a in ranking if a in legal_actions), None)
                            dec = fallback if fallback else ('hit' if 'hit' in legal_actions else 'stand')

                        if dec == 'hit':
                            self.draw()
                            if self.verbose:
                                print(self.__str__())
                            self.check_busted()
                            if self.busted:
                                break
                        elif dec == 'stand':
                            self.done = True
                            break
                        elif dec == 'double_down':
                            self.double_down()
                            self.check_busted()
                            break
                        elif dec == 'split_pairs':
                            self.split_pairs()
                            break
                        elif dec == 'surrender':
                            self.surrender()
                            break
                        elif dec == 'insurance':
                            self.insure()
                        else:
                            print('Please Enter a Valid Decision (Hit or stand)')

        if not (dealer and round_number == 0):
            if self.verbose:
                print(self.__str__())
        self.check_busted()
        if self.busted:
            if self.verbose:
                print(self.name, 'has busted and their turn is skipped')
            self.done = True

    def check_busted(self):
        """
        Check if the player is busted
        """
        if self.points > 21:
            self.busted = True
            
    def make_wager(self, amount=None):
        """
        Make a wager. Amount is subtracted from bankroll and added to current wager. 
        Check if amount is allowed to be wagered, if not default to 0

        Args:
            amount (float): Amount to be wagered.
        """
        if amount == None:
            amount = self.default_wager
        if self.is_dealer:
            amount = 0
        if self.eligible_wager(amount):
            self.wager += amount
            self.bankroll -= amount
            if self.verbose:
                print(f'Player {self.name} has bet {amount} and now has {self.bankroll}')
        else:
            amount = 0
            if self.verbose:
                print(f'Player {self.name} does not have enough money to bet {amount} (only {int(self.bankroll)}), so wager set to 0')

    def eligible_wager(self, amount):
        """
        Check if a wager is eligible
        Args:
            amount (float): Amount to be checked if it's allowed
        """
        if amount <= self.bankroll:
            return True
        else:
            if self.verbose:
                print('WAGER IS NOT ELIGIBLE')
            return False
        
    def calc_winnings(self, result):
        """
        Calculate the amount to be awarded to the bankroll
        Args:
            result (int): Result of the game
        """
        profit = self.wager * result
        win_lose = self.wager + profit  # return original wager + profit (or original wager - loss)
        if self.verbose:
            # Use root_player to print the correct total wallet balance
            print(f'Player {self.name} has won or lost {win_lose - self.wager} and now has {self.root_player.bankroll+win_lose}')
        self.wager = 0  
        return win_lose

    def reset(self):
        """
        Reset the player relavent variables after each round
        """
        self.hand = []
        self.points = 0
        self.busted = False
        self.done = False
        self.wager = 0
        self.surrendered = False
        self.double_downed = False
        self.splitted = False
        self.insured = False
        self.history = []

    def adjust_wager(self):
        """
        Take the true count and apply the Kelly criterion to get a new wager as a percentage of the current bankroll.
        """
        true_count = self.deck.true_count
        if not self.is_dealer:
            f = max(0, (true_count - 1) / 200)
            if self.bankroll * f >= self.min_bet:
                new_wager = round(self.bankroll * f, 2)
                if new_wager > self.max_bet:
                    self.default_wager = self.max_bet
                else:
                    self.default_wager = new_wager
            else:
                self.default_wager = self.min_bet
        else:
            pass          
    
    def surrender(self):
        """
        Surrender: forfeit your hand and only lose 50% of your wager
        """
        self.done = True
        self.surrendered = True
        if self.verbose:
            print(f'{self.name} has surrendered and forfeits half of their bet ({self.wager/2})')

    def double_down(self):
        """
        Double down: Bet 100% of your wager again, draw a card, and end your turn.
        """
        if self.verbose:
            print(f'{self.name} has doubled down and will take another card and double their wager to {self.wager*2}')
        self.root_player.bankroll -= self.wager
        self.wager *= 2
        self.draw()
        self.done = True
        self.double_downed = True

    def insure(self):
        """
        Insure: If the dealer up card is an Ace, you may bet up to 50% of your initial wager (this function always bets 50%) in a side pot
        If the dealer has a natural blackjack, you lose your initial wager and win the side pot.
        """
        self.insurance_bet = self.wager * 0.5
        self.root_player.bankroll -= self.insurance_bet
        self.insured = True

    def split_pairs(self):
        """
        Split pairs: If you have two of the same card, you may "split" and play two seperate hands that both have your initial wager.
        """
        new_name = self.name + '_split'
        dec_func = self.dec_func
        
        hand = [self.hand[1]]
        self.hand.pop()
        verbose = self.verbose
        min_bet = self.min_bet
        count = self.count_cards_val
        initial_buyin = 0
        
        player_2 = Player(name=new_name, initial_buyin=initial_buyin, dec_func=dec_func, verbose=verbose, min_bet=min_bet, count_cards=count)
        
        # LINK TO ORIGINAL WALLET
        player_2.parent = self.root_player 

        player_2.hand = hand
        player_2.wager = self.wager
        
        # DEDUCT FROM ROOT BANKROLL
        self.root_player.bankroll -= player_2.wager 
        
        player_2.deck = self.deck

        self.draw()
        player_2.draw()

        self.points = self.calc_points()
        player_2.points = player_2.calc_points()

        self.split_obj = player_2
        self.splitted = True
        if self.verbose:
            print(f'Player has split {self.hand[0]} with a {self.hand[1]}')
   
        

IndentationError: unindent does not match any outer indentation level (<string>, line 156)

In [ ]:
class Qtable():
    """
    Blackjack Qtable. Trained by the trainer class. Contains all possible states along with all action values. Keys are states and values are action.
    The Qtable can be made into a decision function. Contains methods to save, load, and print table as well.
    Attributes:
        params (list): List of parameters. First value is the training rate and second is the discount factor

    """
    def __init__(self, params=[0.008, 0.95]): # learning rate and discount factor
        self.params = params
        self.actions = ['hit', 'stand', 'surrender', 'insurance', 'double_down', 'split_pairs']
        self.q = defaultdict(lambda: {'hit':0, 'stand':0,'surrender':0,'insurance':0,'double_down':0,'split_pairs':0})
    def best_action(self, state, legal_actions):
        """
        Chooses the best action based on the state of the game. Chooses from the legal actions.
        Args:
            state (list): The current state of the game. Includes points, dealer up card, useable aces (bool), and true count
            bucketed -5 to 5.
            legal_actions (list): A list of legal actions to choose from
        """
        # state = (points, dealer_card, ace)
        points = state[0]
        dealer_card = state[1]
        ace = state[2]
        true_count = state[3]
        values = self.q[points, dealer_card, ace, true_count]
        values = {action: values[action] for action in self.actions if action in legal_actions}
        return max(values, key=values.get)
    def update(self, state, next_state, reward, action):
        """
        Uses bellmans update formula to update the action value for a state key.
        Args:
            state (list): The current state of the game. Includes points, dealer up card, useable aces (bool), and true count
            bucketed -5 to 5.
            next_state (list): Same as state. Tracks intermediate rewards
            reward (int): Result from a round. Used to reward model
            action (string): Action the model chose
            
        """
        alpha = self.params[0]
        beta = self.params[1]
        future_best = 0
        if next_state:
            future_vals = self.q[next_state].values()
            future_best = max(future_vals)
        self.q[state][action] += alpha * (beta*future_best + reward - self.q[state][action])
    def print_table(self):
        """
        Print the trained Qtable
        """
        print("\n--- Final Q-Table ---")
        for state, actions in sorted(self.q.items()):
            hit_val = actions['hit']
            stand_val = actions['stand']
            surrender_val = actions['surrender']
            insure_val = actions['insurance']
            double_val = actions['double_down']
            split_val = actions['split_pairs']
            
            
            print(f"State: {state} | Hit: {hit_val:+.3f} | Stand: {stand_val:+.3f} Surrender: {surrender_val:+.3f} | Insurance: {insure_val:+.3f}| Double Down: {double_val:+.3f} | Split Pairs: {split_val:+.3f}")
    def get_table(self):
        """
        Return a trained Qtable
        """
        return self.q

    def save_table(self):
        """
        Saved a trained Qtable into a pickle file
        """
        with open('saved_qtable.pkl', 'wb') as f:
            pk.dump(self.q, f)
        print('Qtable saved to "saved_qtable.pkl"')
    @classmethod
    def load_table(cls, file_name='saved_qtable.pkl'):
        """
        Load a trained Qtable (using dill)
        """
        with open(file_name,'rb') as d:
            loaded_qtable = pk.load(d)
        return loaded_qtable

In [ ]:
class Trainer():
    """
    Class used to train the Qtable using a reinforcement learning model.
    Attributes:
        hyperparams (list): A list in the following order
            [eps_start, eps_end, eps_decay, episodes/iterations]
            
            eps (epsilon) is the curiosity factor. The higher the curiosity the more the model explores all the possible actions. Important to build full
            Qtable and make sure all actions are tested for each state.
            
            eps_start (float): Starting eps value. Should be 1 or very close to 1.
            eps_end (float): Ending eps value. Model will not decay past this value.
            eps_decay (float): How much model decays after every episode.
            episodes (int): How many rounds of Blackjack the model plays.

        game_class (class): Name of the Blackjack class.
        player_class (class): Name of the Player class.
        player_name (str): Name of the player.
        dealer_name (str): Name of the dealer.
        silent (bool): Turns on and off verbosity in player and model class. Recommended True.
        Wagers (bool): Turns on and off wagers in player class.





    """
    def __init__(self, hyperparams = [1, 0.03, 0.999, int(10000)], game_class=Blackjack, player_class=Player, player_name='John', dealer_name='dealer', silent=True, Wagers=False):
        self.game = game_class
        self.player = player_class
        self.player_name = player_name
        self.dealer_name = dealer_name
        self.qtable = Qtable()
        self.history = []
        self.eps_start = hyperparams[0]
        self.eps_end = hyperparams[1]
        self.eps_decay = hyperparams[2]
        self.episodes = hyperparams[3]
        self.eps = self.eps_start
        self.wagers = Wagers

        self.last_action = None
        self.last_state = None
        self.dealer = player_class(name='d', is_dealer=True)
        self.player = player_class(is_dealer=False, dec_func=self.decision_function)
        self.game = game_class([self.dealer, self.player], wagers=self.wagers)

    def decay_eps(self):
        """
        Decay eps until reaching eps_end
        """
        if self.eps > self.eps_end:
            self.eps *= self.eps_decay
   
    def decision_function(self, player_obj, points, hand, dealer_card, true_count):
        """
        The function the model uses to make decisions in a game based on the Qtable. Buckets true count in range(-5, 6).
        
        Args:
            player_obj (class): The instance of the player/model that is playing.
            points (int): Points for the hand.
            dealer_card (str or int): Dealer up card.
            true_count (float): The current true count.
        """
        legal_actions = self.game.get_legal_actions(self.player)
        tc_bucket = max(-5, min(5, round(true_count)))
        useable_ace = self.check_useable_ace(hand)
        if dealer_card in ['J', 'Q', 'K']:
            dealer_card = 10
        elif dealer_card == 'Ace':
            dealer_card = 11 
        else:
            dealer_card = dealer_card
        current_state = (points, dealer_card, useable_ace, tc_bucket)
        
            
        if random.random() > (self.eps):
            action = self.qtable.best_action(current_state, legal_actions)
        else:
            action = random.choice(legal_actions)
            
        self.last_state = current_state
        self.last_action = action
        self.history.append((current_state, action))
        player_obj.history.append((current_state, action))
        return action
       
        
    def run_round(self):
        """
        Runs a round of blackjack and updates the model based on the result of a round. Backpropogates through the hand history to get all
        intermediate actions, which is needed for the bellman update formula. Decay epsilon at the end of a round.
        """
        winners, rewards = self.game.play_round()
        
        for i, player_obj in enumerate(self.game.players[1:]):
            reward = rewards[i]
            next_state = None
            
            for state, action in reversed(player_obj.history):
                self.qtable.update(state, next_state, reward, action)
                next_state = state
                reward = 0

        self.decay_eps()
        
    #def update_from_history(self, reward):
     #
        #future_state = None
#
 #       for state, action in zip(reversed(self.states), reversed(self.decisions)):
  #          self.qtable.update(state, future_state, reward, action)
   #         future_state = state
    #        reward = 0

    def check_useable_ace(self, hand):
        """
        Helper function to check if a hand has a useable ace.
        Args:
            hand (list): The current hand to be checked
        Return (bool): True if there is a useable ace, false otherwise
        """
        if 'Ace' not in hand:
            return False
        hard_points = 0
        for card in hand:
            if card in ['J', 'Q', 'K']:
                hard_points += 10
            elif card == 'Ace':
                hard_points += 1
            else:
                hard_points += card
        return hard_points <= 11

    def train(self):
        """
        Train the model for the specified amount of episodes and print the resulting Qtable.
        """
        print(f'Training {self.episodes} episodes')
        for e in range(self.episodes):
            self.run_round()
            if e % 20000 == 0:
                print(f'Episode {e} complete. Epsilon: {self.eps:.3f}')
        
        print('Training complete!')
        self.qtable.print_table()
        

In [ ]:
class Deck():
    """
    Class that handles the deck. Keeps track of true count and card counting. By default uses wong halves, but can use HiLo system as well. 
    Attributes:
        shoes (int): Number of decks used in a game before being reset. Important for card counting.
        count (bool): Whether to count cards or not
    """
    def __init__(self, shoes=6, count=True):
        self.digits = list(range(2,11))
        self.faces = ['K', 'Q', 'J', 'Ace']
        self.cards = self.digits + self.faces
        self.shoes = shoes
        self.deck = []
        for s in range(self.shoes):
            for c in self.cards:
                for _ in range(4):
                    self.deck.append(c)
        random.shuffle(self.deck)
        self.drawn = []
        self.deck_size = len(self.deck)
        self.decks_remaining = self.deck_size / 52
        self.reset = False
        self.count = count
        self.running_count = 0
        self.true_count = 0

        self.round_draw = []
        self.wong_halves_ = {'Ace':-1, 2:0.5, 3:1, 4:1, 5:1.5, 6:1, 7:0.5, 8:0, 9:-0.5, 10:-1}
        self.hilo_ = {'Ace':-1, 2:1, 3:1, 4:1, 5:1, 6:1, 7:0, 8:0, 9:0, 10:-1,}

    
    def draw(self, count):
        """
        Draw a card from the deck.
        Args:
            count (bool). Whether to call count_cards method.
        Returns (str or int): The drawn card.
        """
        if self.deck_size == 0:
            self.reset_deck()
        choice = self.deck[-1]
        self.drawn.append(choice)
        if self.count and count:
            self.count_cards(choice)
        self.deck.pop()
        self.deck_size -= 1
        self.decks_remaining = self.deck_size / 52
        return choice

    def reset_deck(self):
        """
        Reset the deck, count, and drawn list.
        """
        self.deck = []
        for s in range(self.shoes):
            for c in self.cards:
                for _ in range(4):
                    self.deck.append(c)
        random.shuffle(self.deck)
        self.running_count = 0
        self.true_count = 0
        self.drawn = []
        self.reset = True
        self.deck_size = len(self.deck)
        self.decks_remaining = self.deck_size / 52
        
    def count_cards(self, card, system='wong_halves'):
        """
        Count cards and keep running and true count. 
        
        Args:
            card (str or int): Card to be counted
            system (str): Which card counting system to be used ("wong_halves" or "hilo")
        """
        if system.lower().strip() == 'wong_halves':
            strategy = self.wong_halves_
        elif system.lower().strip() == 'hilo':
            strategy = self.hilo_
        val = 10 if card in ['K', 'Q', 'J'] else card
        self.running_count += strategy[val]
        self.true_count = self.running_count / max(1, self.decks_remaining)
        
    
        

In [ ]:
class Evaluator:
    """
    Class used to evaluate a models performance. Runs the model over thousands of rounds and takes final result and graphs progression of bankroll.

    Attributes:
        Qtable (collections defaultdict object): Trained Qtable
        game_class (class): Name of Blackjack class.
        player_class (class): Name of player class
        trained (class): Name of trainer class
        rounds (int): How many rounds the model will play for
        epochs (int): How many times the model will play the rounds for, to smooth data


    """
    def __init__(self, qtable, game_class=Blackjack, player_class=Player, trainer=Trainer, epochs=50, rounds=5000):
        self.epochs = epochs
        self.q = qtable
        self.bankroll_history = []
        self.times_surrendered = 0
        self.times_hit = 0
        self.times_stand = 0
        self.times_insured = 0
        self.times_double = 0
        self.times_split = 0
        self.rounds = rounds
        self.player_class = player_class
        self.game_class = game_class
        trainer_i = trainer()

        def get_dec_func(trainer):
            qtable = self.q
            def trained_dec(player_obj, points, hand, dealer_card, true_count):
                if dealer_card in ['J', 'Q', 'K']:
                    dealer_card_val = 10
                elif dealer_card == 'Ace':
                    dealer_card_val = 11
                else:
                    dealer_card_val = int(dealer_card)

                useable_ace = trainer.check_useable_ace(hand)
                tc_bucket = max(-5, min(5, round(true_count)))
                state = (points, dealer_card_val, useable_ace, tc_bucket)
                values = qtable[state]

                if not values:
                    return 'stand'
                best = max(values, key=values.get)
                # store ranked actions so the safeguard in Player.turn() can use them
                trained_dec._last_action_ranking = sorted(values, key=values.get, reverse=True)
                return best
            return trained_dec

        self.trained_dec_func = get_dec_func(trainer_i)
        self.player = player_class(dec_func=self.trained_dec_func, verbose=False, count_cards=False)
        self.dealer = player_class(is_dealer=True, verbose=False)
        players = [self.dealer, self.player]
        self.game = game_class(players)
        self.final_bankrolls = []
        self.winvals = 0
        self.wager_count = 0

    def eval(self):
        """
        Run the model for the specificed amount of rounds over the specified number of epochs. Save bankrolls and bankroll history. Track how many 
        times the model does each special action. Track how many times the model wins rounds (not 100% accurate, can be ignored).
        """
        for e in range(self.epochs):
            bankrolls = []
            self.player = self.player_class(verbose=False, count_cards=True, dec_func=self.trained_dec_func)
            self.dealer = self.player_class(verbose=False, is_dealer=True)
            self.game = self.game_class([self.dealer, self.player], do_insurance=False, do_double_down=True, do_split_pairs=True)
            for r in range(self.rounds):
                winval = self.game.play_round()[0]
                self.winvals += winval[0]
                bankrolls.append(self.player.bankroll)
                if self.player.surrendered:
                    self.times_surrendered += 1
                if self.player.insured:
                    self.times_insured += 1
                if self.player.double_downed:
                    self.times_double += 1
                if self.player.splitted:
                    self.times_split += 1
            self.final_bankrolls.append(self.player.bankroll)
            self.bankroll_history.append(bankrolls)
            
    def get_data(self):
        """
        Displays data from the evaluation. Prints average final bankrolls, number of times special actions were taken, and a graph of the
        average bankroll progression over the epochs. 
        """
        self.avg_wins = self.winvals / (self.rounds * self.epochs)
        print('BLACKJACK PLAYER SUMMARY ----------------------------------------------------------\n\n')
        print('On average, the model won', self.avg_wins)
        print(f'The model played {self.rounds} rounds {self.epochs} times')
        final_avg = np.mean(np.array(self.final_bankrolls))
        print('The average final ending amount for the model was', final_avg)
        print(f'The model surrendered {self.times_surrendered} times')
        print(f'The model double_downed {self.times_double} times')
        print(f'The model split {self.times_split} times')
        print(f'The model insured {self.times_insured} times')
        bank = np.array(self.bankroll_history)
        bank_avg = []
        for i in range(len(bank[0])):
            bank_avg.append(np.mean(bank[:, i]))
        plt.plot(range(self.rounds), bank_avg)
        plt.ylabel('Bankroll Average')
        plt.xlabel('Round #')
        plt.title('Average bankroll per round')


In [ ]:
'''
def random_hit(points, hand, dealer_card, true_count):
    rand = random.random()
    if rand >= 0.5:
        return 'hit'
    if rand < 0.5:
        return 'stand'

def basic_dec(points, hand, dealer_card, true_count):
    if points >= 16:
        return 'stand'
    elif points < 16:
        return 'hit'
'''

In [ ]:
'''
player = Player(verbose=True, count_cards=True)
dealer = Player(name='Dealer', is_dealer=True, verbose=True)
players = [dealer, player]
game = Blackjack(players)
for _ in range(100):
    print('GAME NUMBER', _ ,'---------------------------------------------')
    game.play_round()
'''

In [ ]:
'''
min_bet = 10
dealer = Player(name='Dealer', is_dealer=True)
player = Player(name='Josh', dec_func=basic_dec, count_cards=True)
player1 = Player(name='Joe', dec_func=random_hit)
player2 = Player(name='Albert', dec_func=basic_dec, count_cards=False, verbose=True)
players = [dealer, player, player1, player2]
game = Blackjack(players)
for _ in range(100):
    game.play_round()
'''


In [ ]:
'''
def random_hit(points, hand, dealer_card, true_count):
    rand = random.random()
    if rand >= 0.5:
        return 'hit'
    if rand < 0.5:
        return 'stand'

def basic_dec(points, hand, dealer_card, true_count):
    if points >= 16:
        return 'stand'
    elif points < 16:
        return 'hit'
def smarter_dec(points, hand, dealer_card, true_count):
    dealer_vals = {
        'Ace': 11,
        'K': 10,
        'Q': 10,
        'J': 10
    }

    dealer_points = dealer_vals.get(dealer_card, dealer_card)

    soft = 'Ace' in hand and points <= 21 and points - 11 + 1 <= 10

    if soft:
        if points <= 17:
            return 'hit'
        elif points == 18:
            if dealer_points >= 9:
                return 'hit'
            return 'stand'
        else:
            return 'stand'

    if points <= 11:
        return 'hit'
    elif 12 <= points <= 16:
        if dealer_points >= 7:
            return 'hit'
        else:
            return 'stand'
    else:
        return 'stand'
'''

In [ ]:
'''
eps_0 = 1
eps_f = 0.02
eps_decay = 0.999999
episodes = 9500000
hypers = [eps_0, eps_f, eps_decay, episodes]
trainer = Trainer(hyperparams=hypers)
trainer.train()
q_table = trainer.qtable.get_table()
trainer.qtable.save_table()
'''

In [ ]:
'''
qtable = Qtable()
q_table = qtable.load_table(file_name='saved_qtable.pkl') # to get saved q table
#q_table = trainer.qtable.get_table()
evaluator = Evaluator(qtable=q_table, rounds=100000, epochs=10)
evaluator.eval()
evaluator.get_data()
'''